
# import required libraries for NLP preprocessing and data manipulation

In [1]:


import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [2]:
# Download required NLTK data (runs once) 

def download_nltk_resources():
    resources = ["punkt", "stopwords", "wordnet", "omw-1.4", "punkt_tab"]
    for resource in resources:
        nltk.download(resource, quiet=True)

# text cleaning using NLP

In [3]:
def to_lowercase(text: str) -> str:
    """Convert all characters to lowercase."""
    return text.lower()

In [4]:

def remove_html_tags(text: str) -> str:
    """Strip any HTML tags that may have been scraped."""
    return re.sub(r"<[^>]+>", " ", text)

In [5]:

def remove_urls(text: str) -> str:
    """Remove any URLs."""
    return re.sub(r"http\S+|www\S+", " ", text)

In [6]:
def remove_punctuation(text: str) -> str:
    """Remove punctuation and special characters, keep letters and spaces."""
    return re.sub(r"[^a-z\s]", " ", text)

In [7]:
def remove_extra_spaces(text: str) -> str:
    """Collapse multiple spaces into one and strip leading/trailing spaces."""
    return re.sub(r"\s+", " ", text).strip()

In [8]:
def tokenize(text: str) -> list[str]:
    """Split text into individual word tokens."""
    return word_tokenize(text)

In [9]:
def remove_stopwords(tokens: list[str]) -> list[str]:
    """Remove common English stopwords (the, is, a, an, …)."""
    stop_words = set(stopwords.words("english"))
    return [token for token in tokens if token not in stop_words]

In [10]:
def lemmatize(tokens: list[str]) -> list[str]:
    """
    Reduce each word to its base form.
    e.g. 'running' → 'run', 'battles' → 'battle'
    """
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(token) for token in tokens]

In [11]:
def remove_short_words(tokens: list[str], min_length: int = 2) -> list[str]:
    """Drop single-character tokens that add no meaning."""
    return [token for token in tokens if len(token) >= min_length]

# full pipeline

In [ ]:

def clean_storyline(text: str) -> str:
    """
    Apply the complete cleaning pipeline to a single storyline string.
    Returns a clean, space-joined string ready for vectorization.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    text = to_lowercase(text)
    text = remove_html_tags(text)
    text = remove_urls(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)

    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    tokens = remove_short_words(tokens)

    return " ".join(tokens)

# Analysis Helper

In [13]:
def show_sample_comparison(df: pd.DataFrame, n: int = 3):
    """Print a before/after comparison for a few rows."""
    print("\n── Sample before / after cleaning ──────────────────────────────")
    for _, row in df.head(n).iterrows():
        print(f"\nMovie  : {row['Movie Name']}")
        print(f"Before : {row['Storyline'][:120]}…")
        print(f"After  : {row['Cleaned_Storyline'][:120]}…")
    print()


def show_basic_stats(df: pd.DataFrame):
    """Print dataset statistics after cleaning."""
    total      = len(df)
    with_plot  = df["Cleaned_Storyline"].apply(lambda x: len(x) > 0).sum()
    empty      = total - with_plot
    avg_words  = df["Cleaned_Storyline"].apply(lambda x: len(x.split())).mean()

    print("── Dataset statistics ──────────────────────────────────────────")
    print(f"  Total movies       : {total}")
    print(f"  With storyline     : {with_plot}")
    print(f"  Empty storylines   : {empty}  (will be dropped)")
    print(f"  Avg words (cleaned): {avg_words:.1f}")
    print()

# Main function

In [14]:
def main():
    INPUT_FILE  = "imdb_movies_2024.csv"
    OUTPUT_FILE = "preprocessed_movies.csv"

    print("Phase 2 — NLP Preprocessing\n")

    # 1. Download NLTK data
    print("Downloading NLTK resources…")
    download_nltk_resources()

    # 2. Load raw data
    print(f"Loading '{INPUT_FILE}'…")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print(f"ERROR: '{INPUT_FILE}' not found. Run scraper.py first.")
        return

    print(f"  Loaded {len(df)} rows with columns: {list(df.columns)}\n")

    # 3. Basic validation
    required_cols = {"Movie Name", "Storyline"}
    if not required_cols.issubset(df.columns):
        print(f"ERROR: CSV must have columns {required_cols}. Found: {list(df.columns)}")
        return

    # 4. Drop rows with missing data
    before = len(df)
    df.dropna(subset=["Movie Name", "Storyline"], inplace=True)
    df.drop_duplicates(subset="Movie Name", inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"  Dropped {before - len(df)} duplicate/empty rows → {len(df)} remain\n")

    # 5. Apply cleaning pipeline
    print("Cleaning storylines…")
    df["Cleaned_Storyline"] = df["Storyline"].apply(clean_storyline)

    # 6. Drop rows where cleaning left an empty string
    df = df[df["Cleaned_Storyline"].str.strip() != ""]
    df.reset_index(drop=True, inplace=True)

    # 7. Show stats and samples
    show_basic_stats(df)
    show_sample_comparison(df)

    # 8. Save output
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    print(f"Saved cleaned data to '{OUTPUT_FILE}'")
    print(df[["Movie Name", "Cleaned_Storyline"]].head())


if __name__ == "__main__":
    main()


Phase 2 — NLP Preprocessing

Loading 'imdb_movies_2024.csv'…
  Loaded 5099 rows with columns: ['Movie Name', 'Storyline']

  Dropped 40 duplicate/empty rows → 5059 remain

Cleaning storylines…
── Dataset statistics ──────────────────────────────────────────
  Total movies       : 5059
  With storyline     : 5059
  Empty storylines   : 0  (will be dropped)
  Avg words (cleaned): 19.0


── Sample before / after cleaning ──────────────────────────────

Movie  : The Substance
Before : A fading celebrity takes a black-market drug: a cell-replicating substance that helps her create a younger, better versi…
After  : fading celebrity take black market drug cell replicating substance help create younger better version…

Movie  : The Life of Chuck
Before : A life-affirming, genre-bending story about three chapters in the life of an ordinary man named Charles Krantz.…
After  : life affirming genre bending story three chapter life ordinary man named charles krantz…

Movie  : Bone Lake
Before : A c

Required package for machine learing


# TF-IDF Model create

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

from recommender import display_recommendations
class MovieRecommender:
    """
    Builds a TF-IDF matrix from cleaned storylines and returns the top-N
    most similar movies for any input storyline.
    """

    def __init__(self,
                 max_features: int = 5000,
                 ngram_range: tuple = (1, 2),
                 top_n: int = 5):
        """
        Parameters
        ----------
        max_features : int
            Maximum number of unique terms to keep in the vocabulary.
            Higher = richer model but more memory. 5000 is a good default.
        ngram_range : tuple
            (1, 1) → single words only
            (1, 2) → single words + two-word phrases (recommended)
        top_n : int
            Number of recommendations to return.
        """
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            sublinear_tf=True,   # apply log(1+tf) to dampen common terms
        )
        self.top_n      = top_n
        self.tfidf_matrix = None
        self.df           = None

    # ── Training / fitting ────────────────────────────────────────────────────

    def fit(self, df: pd.DataFrame):
        """
        Fit the TF-IDF vectorizer on the cleaned storylines and store the
        resulting matrix.

        Parameters
        ----------
        df : DataFrame with columns 'Movie Name' and 'Cleaned_Storyline'
        """
        self.df = df.reset_index(drop=True)
        print(f"Fitting TF-IDF on {len(df)} movies…")
        self.tfidf_matrix = self.vectorizer.fit_transform(
            self.df["Cleaned_Storyline"]
        )
        print(f"  Vocabulary size : {len(self.vectorizer.vocabulary_)}")
        print(f"  Matrix shape    : {self.tfidf_matrix.shape}")  # (movies, terms)

    # ── Recommendation ────────────────────────────────────────────────────────

    def recommend(self, input_storyline: str) -> pd.DataFrame:
        """
        Given a raw or cleaned input storyline, return the top-N similar movies.

        Parameters
        ----------
        input_storyline : str
            Free-text plot description entered by the user.

        Returns
        -------
        DataFrame with columns: Rank, Movie Name, Storyline, Similarity Score
        """
        if self.tfidf_matrix is None:
            raise RuntimeError("Call fit() before recommend().")

        # Clean the input the same way the training data was cleaned
        cleaned_input = clean_storyline(input_storyline)
        if not cleaned_input:
            print("Warning: input storyline is empty after cleaning.")
            return pd.DataFrame()

        # Transform input into TF-IDF space
        input_vector  = self.vectorizer.transform([cleaned_input])

        # Compute cosine similarity against every movie in the matrix
        similarity_scores = cosine_similarity(input_vector, self.tfidf_matrix)[0]

        # Rank movies by similarity score (descending)
        top_indices = similarity_scores.argsort()[::-1][:self.top_n]

        results = []
        for rank, idx in enumerate(top_indices, start=1):
            results.append({
                "Rank"             : rank,
                "Movie Name"       : self.df.loc[idx, "Movie Name"],
                "Storyline"        : self.df.loc[idx, "Storyline"],
                "Similarity_Score" : round(float(similarity_scores[idx]), 4),
            })

        return pd.DataFrame(results)

    # ── Movie-to-movie recommendations ───────────────────────────────────────

    def recommend_by_title(self, movie_title: str) -> pd.DataFrame:
        """
        Find movies similar to a title that already exists in the dataset.

        Parameters
        ----------
        movie_title : str  (case-insensitive, partial match supported)
        """
        if self.df is None:
            raise RuntimeError("Call fit() first.")

        mask = self.df["Movie Name"].str.lower().str.contains(
            movie_title.lower(), na=False
        )
        matches = self.df[mask]

        if matches.empty:
            print(f"No movie found matching '{movie_title}'.")
            return pd.DataFrame()

        # Use the first matching movie's cleaned storyline as the query
        idx        = matches.index[0]
        title_used = self.df.loc[idx, "Movie Name"]
        print(f"Using movie: '{title_used}'")

        input_vector      = self.tfidf_matrix[idx]
        similarity_scores = cosine_similarity(input_vector, self.tfidf_matrix)[0]
        similarity_scores[idx] = -1  # exclude the query movie itself

        top_indices = similarity_scores.argsort()[::-1][:self.top_n]

        results = []
        for rank, i in enumerate(top_indices, start=1):
            results.append({
                "Rank"             : rank,
                "Movie Name"       : self.df.loc[i, "Movie Name"],
                "Storyline"        : self.df.loc[i, "Storyline"],
                "Similarity_Score" : round(float(similarity_scores[i]), 4),
            })

        return pd.DataFrame(results)

    # ── Persistence ───────────────────────────────────────────────────────────

    def save(self, path: str = "recommender.pkl"):
        """Pickle the fitted model for reuse in the Streamlit app."""
        with open(path, "wb") as f:
            pickle.dump(self, f)
        print(f"Model saved to '{path}'")

    @staticmethod
    def load(path: str = "recommender.pkl") -> "MovieRecommender":
        """Load a previously saved model."""
        with open(path, "rb") as f:
            model = pickle.load(f)
        print(f"Model loaded from '{path}'")
        return model


# ── Evaluation helper ─────────────────────────────────────────────────────────

def display_recommendations(results: pd.DataFrame):
    """Pretty-print the recommendations table."""
    if results.empty:
        print("No recommendations found.")
        return

    print("\n── Top Recommendations ──────────────────────────────────────────")
    for _, row in results.iterrows():
        print(f"\n  #{row['Rank']}  {row['Movie Name']}  "
              f"(score: {row['Similarity_Score']})")
        storyline = row["Storyline"]
        if isinstance(storyline, str) and storyline.strip():
            preview = storyline[:150].rstrip() + ("…" if len(storyline) > 150 else "")
            print(f"      {preview}")
    print()


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    INPUT_FILE  = "preprocessed_movies.csv"
    MODEL_FILE  = "recommender.pkl"

    print("Phase 3 — TF-IDF Recommendation Engine\n")

    download_nltk_resources()

    # 1. Load preprocessed data
    print(f"Loading '{INPUT_FILE}'…")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print(f"ERROR: '{INPUT_FILE}' not found. Run preprocessor.py first.")
        return

    required = {"Movie Name", "Storyline", "Cleaned_Storyline"}
    if not required.issubset(df.columns):
        print(f"ERROR: CSV must have columns {required}. Found: {list(df.columns)}")
        return

    df.dropna(subset=["Cleaned_Storyline"], inplace=True)
    df = df[df["Cleaned_Storyline"].str.strip() != ""]
    df.reset_index(drop=True, inplace=True)
    print(f"  {len(df)} usable movies loaded\n")

    # 2. Build and fit the recommender
    recommender = MovieRecommender(
        max_features=5000,
        ngram_range=(1, 2),
        top_n=5,
    )
    recommender.fit(df)

    # 3. Save the model (used by app.py in Phase 5)
    recommender.save(MODEL_FILE)

    # 4. Demo — recommend by custom storyline
    test_storyline = (
        "A young hero discovers magical powers and must battle a dark villain "
        "to save the world, with the help of loyal friends."
    )
    print(f"\nTest query:\n  \"{test_storyline}\"\n")
    results = recommender.recommend(test_storyline)
    display_recommendations(results)

    # 5. Demo — recommend by existing movie title
    print("Recommend movies similar to 'Dune':")
    results2 = recommender.recommend_by_title("Dune")
    display_recommendations(results2)


if __name__ == "__main__":
    main()

Phase 3 — TF-IDF Recommendation Engine

Loading 'preprocessed_movies.csv'…
  5059 usable movies loaded

Fitting TF-IDF on 5059 movies…
  Vocabulary size : 5000
  Matrix shape    : (5059, 5000)
Model saved to 'recommender.pkl'

Test query:
  "A young hero discovers magical powers and must battle a dark villain to save the world, with the help of loyal friends."


── Top Recommendations ──────────────────────────────────────────

  #1  Saturn  (score: 0.2343)
      When a mysterious planet appears in the sky, a young father must choose between the life he loves and an ancient call to save the world.

  #2  Doraemon the Movie: Nobita's Earth Symphony  (score: 0.2283)
      Doraemon and friends go on an adventure to meet new buddies, connect to people with music, and save the world from a crisis.

  #3  Robin and the Hoods  (score: 0.2273)
      For the tenacious 11-year-old Robin and her loyal band of friends 'The Hoods', the patch of overgrown scrubland at the end of their cul-de-sac is 